# octave-webfig 데모

출력 셀 안에서 figure를 직접 만져보는 예제. 맨 위 셀부터 차례로 실행해.

In [ ]:
%webfig on

## 1. 2-DOF 로봇팔 GUI (슬라이더 콜백)
슬라이더를 끌면 Octave 세션에서 `update_arm_plot` 이 실제로 실행되고 그림이 갱신돼.

In [ ]:
function update_arm_plot(src, ~)
  h = guidata(src);
  t1 = get(h.theta1_slider, 'Value');
  t2 = get(h.theta2_slider, 'Value');
  x1 = h.L1*cos(t1);         y1 = h.L1*sin(t1);
  x2 = x1 + h.L2*cos(t1+t2); y2 = y1 + h.L2*sin(t1+t2);
  set(h.arm_plot, 'XData', [0 x1 x2], 'YData', [0 y1 y2]);
  set(h.end_effector, 'XData', x2, 'YData', y2);
  set(h.theta1_text, 'String', sprintf('%5.1f', rad2deg(t1)));
  set(h.theta2_text, 'String', sprintf('%5.1f', rad2deg(t2)));
  set(h.pos_text, 'String', sprintf('End effector: (%.3f, %.3f)', x2, y2));
end

function robot_arm_gui()
  L1 = 1.0; L2 = 0.8; lim = (L1 + L2) * 1.1;
  h = struct('L1', L1, 'L2', L2);
  h.fig = figure('Position', [100, 100, 600, 500], 'Name', '2-DOF Robot Arm Kinematics', 'NumberTitle', 'off');
  h.ax = axes('Parent', h.fig, 'Units', 'pixels', 'Position', [60, 225, 480, 250], ...
              'XLim', [-lim, lim], 'YLim', [-lim, lim], 'DataAspectRatio', [1 1 1], 'Box', 'on');
  title(h.ax, 'Robot Arm Position'); xlabel(h.ax, 'X position'); ylabel(h.ax, 'Y position');
  grid(h.ax, 'on'); hold(h.ax, 'on');
  h.arm_plot     = plot(h.ax, [0 0 0], [0 0 0], 'b-o', 'LineWidth', 3, 'MarkerSize', 8);
  h.end_effector = plot(h.ax, 0, 0, 'ro', 'MarkerSize', 10, 'MarkerFaceColor', 'r');
  h.base         = plot(h.ax, 0, 0, 'ks', 'MarkerSize', 12, 'MarkerFaceColor', 'k');

  uicontrol(h.fig, 'Style', 'text', 'Position', [30, 160, 100, 20], 'String', 'Theta 1:');
  h.theta1_slider = uicontrol(h.fig, 'Style', 'slider', 'Min', -pi, 'Max', pi, 'Value', 0, ...
                              'Position', [140, 160, 320, 20], 'Callback', @update_arm_plot);
  h.theta1_text = uicontrol(h.fig, 'Style', 'text', 'Position', [480, 160, 50, 20], 'String', '0.0');
  uicontrol(h.fig, 'Style', 'text', 'Position', [30, 120, 100, 20], 'String', 'Theta 2:');
  h.theta2_slider = uicontrol(h.fig, 'Style', 'slider', 'Min', -pi, 'Max', pi, 'Value', 0, ...
                              'Position', [140, 120, 320, 20], 'Callback', @update_arm_plot);
  h.theta2_text = uicontrol(h.fig, 'Style', 'text', 'Position', [480, 120, 50, 20], 'String', '0.0');
  h.pos_text = uicontrol(h.fig, 'Style', 'text', 'Position', [140, 70, 320, 20], 'String', '');
  guidata(h.fig, h);
  update_arm_plot(h.fig);
end

robot_arm_gui();

## 2. drawnow 애니메이션
셀이 도는 동안 출력 셀의 그림이 실시간으로 움직여. (`%webfig fps 30` 으로 최대 프레임 수 조절)

In [ ]:
L1 = 1.0; L2 = 0.8; lim = 2;
figure; hold on; axis equal; axis([-lim lim -lim lim]); grid on;
arm = plot([0 0 0], [0 0 0], 'b-o', 'LineWidth', 3);
trail = plot(NaN, NaN, 'r-');
tx = []; ty = [];
for t = linspace(0, 2*pi, 150)
  t1 = pi/4 * sin(t); t2 = pi/2 * sin(2*t);
  x1 = L1*cos(t1); y1 = L1*sin(t1);
  x2 = x1 + L2*cos(t1+t2); y2 = y1 + L2*sin(t1+t2);
  tx(end+1) = x2; ty(end+1) = y2;
  set(arm, 'XData', [0 x1 x2], 'YData', [0 y1 y2]);
  set(trail, 'XData', tx, 'YData', ty);
  title(sprintf('t = %.2f', t));
  drawnow;
  pause(0.02);
end

## 3. 3D: 드래그로 회전, 휠로 확대, 더블클릭으로 초기화

In [ ]:
[X, Y, Z] = peaks(30);
figure; surf(X, Y, Z); colorbar;
xlabel('x'); ylabel('y'); zlabel('z'); title('peaks');

## 4. 2D: 드래그로 이동, 휠로 확대, Shift+드래그로 영역 확대, 점 위에 마우스를 올리면 좌표

In [ ]:
t = linspace(0, 4*pi, 400);
figure;
subplot(2,1,1); plot(t, sin(t), t, cos(t), '--'); grid on; legend('sin', 'cos'); title('sin / cos');
subplot(2,1,2); semilogy(t, exp(t/3), 'o-', 'MarkerSize', 3); grid on; title('semilogy');

In [ ]:
%webfig status